In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts

### Preparation
- Import clean metadata dataset to align the metadata (columns of adata.obs) for each dataset.
- Import mapping from gene symbols to gene ids to switch between those two as necessary.
- Import list of gene ids and gene symbols/names which CellRanger Filtering would keep, to analyse the overlap with used datasets.


In [2]:
source_adata = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/LCA_v5.h5ad")

In [170]:
clean_metadata = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/query_dataset_scarches_input.h5ad")

In [171]:
# Rename two fibrosis conditions
clean_metadata.obs["condition"] = clean_metadata.obs["condition"].astype("string")
clean_metadata.obs.loc[clean_metadata.obs.disease=="IPF","condition"] = "Endstage lung fibrosis (IPF)"
clean_metadata.obs.loc[clean_metadata.obs.disease=="EAA","condition"] = "Endstage lung fibrosis (EAA)"
clean_metadata.obs["condition"] = clean_metadata.obs["condition"].astype("category")

In [5]:
mapping_gene_ids_symbols = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_ids_to_gene_symbol.csv")

In [6]:
mapping_gene_ids_symbols.rename(columns={"Unnamed: 0":"gene_id"},inplace=True)

In [7]:
mapping_gene_ids_symbols_short = mapping_gene_ids_symbols.sort_values("gene_id").groupby("gene_name", as_index=False).first()

In [8]:
#Rename double gene_name from mapping_gene_ids_symbols and append -1 to second one from each pair
double_index = (mapping_gene_ids_symbols.groupby("gene_name").count()>1)[(mapping_gene_ids_symbols.groupby("gene_name").count()>1)["gene_id"]].index
mask_rename = ~mapping_gene_ids_symbols.gene_id.isin(mapping_gene_ids_symbols_short.gene_id)
mapping_gene_ids_symbols.loc[mask_rename,"gene_name"] = mapping_gene_ids_symbols.loc[mask_rename,"gene_name"] + "-1"

In [9]:
gene_ids_to_keep = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_ids_to_keep_cellrangerbased.txt", header=None)
gene_ids_to_keep.rename(columns={0:"gene_id"},inplace=True)
gene_ids_to_keep = gene_ids_to_keep.gene_id.tolist()

In [10]:
gene_names_to_keep = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_names_to_keep_cellrangerbased.txt", header=None)
gene_names_to_keep.rename(columns={0:"gene_name"},inplace=True)
gene_names_to_keep = gene_names_to_keep.gene_name.tolist()

## 1. Healthy data
Import all raw healthy datasets, check overlap with CellRanger Gene Filtering and Reference data (source_adata) and get clean metadata.

In [178]:
healthy_data = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/LCA_h5ads/Meyer_2021_raw.h5ad")

In [179]:
healthy_data.obs.index = healthy_data.obs.index+"_meyer"

In [180]:
print("# genes to keep: ", np.sum(healthy_data.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(healthy_data.var.index))
print("ratio of genes to keep: ", np.sum(healthy_data.var.index.isin(gene_ids_to_keep))/len(healthy_data.var.index))

# genes to keep:  33694
# genes total:  33694
ratio of genes to keep:  1.0


In [181]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data1.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.91


In [182]:
healthy_data.obs = pd.concat([healthy_data.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(healthy_data.obs.columns):]

## 2. IPF data
Import all raw IPF datasets, check overlap with CellRanger Gene Filtering and Reference data (source_adata) and get clean metadata.

In [228]:
query_data1 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/adams.h5ad")
query_data2 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/haberman.h5ad")
query_data3 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/reyfman_disease.h5ad")
query_data4 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/tsukui.h5ad")

In [229]:
query_data1.var.index=query_data1.var.ensembl

In [185]:
print("# genes to keep: ", np.sum(query_data1.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data1.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data1.var.index.isin(gene_ids_to_keep))/len(query_data1.var.index),2))

# genes to keep:  30924
# genes total:  45947
ratio of genes to keep:  0.67


In [231]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.gene_ids.isin(query_data1.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.94


In [233]:
# Rename gene_names in query_data2 from xxx.1 to xxx-1
query_data2.var["gene_name"] = query_data2.var.index
tmp = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name", how="left")
mask_rename_genes_query2 = query_data2.var.gene_name.isin(tmp[tmp.gene_id.isna()].gene_name)
query_data2.var.loc[mask_rename_genes_query2,"gene_name"] = query_data2.var[mask_rename_genes_query2].gene_name.str[:-2]+"-1"

In [ ]:
query_data2.var = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name")
query_data2.var.index = query_data2.var.gene_id

In [235]:
print("# genes to keep: ", np.sum(query_data2.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data2.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data2.var.index.isin(gene_ids_to_keep))/len(query_data2.var.index),2))

# genes to keep:  33694
# genes total:  33694
ratio of genes to keep:  1.0


In [236]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.gene_ids.isin(query_data2.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  1.0


In [237]:
query_data3.var["gene_name"] = query_data3.var.index
query_data3.var = query_data3.var.merge(mapping_gene_ids_symbols, on="gene_name").copy()
query_data3.var.index = query_data3.var.gene_id

In [238]:
print("# genes to keep: ", np.sum(query_data3.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data3.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data3.var.index.isin(gene_ids_to_keep))/len(query_data3.var.index),2))

# genes to keep:  27181
# genes total:  27181
ratio of genes to keep:  1.0


In [239]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.gene_ids.isin(query_data3.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.93


In [240]:
query_data4.var["gene_name"] = query_data4.var.index
query_data4.var = query_data4.var.merge(mapping_gene_ids_symbols, on="gene_name")
query_data4.var.index = query_data4.var.gene_id

In [241]:
print("# genes to keep: ", np.sum(query_data4.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data4.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data4.var.index.isin(gene_ids_to_keep))/len(query_data4.var.index),2))

# genes to keep:  27147
# genes total:  27147
ratio of genes to keep:  1.0


In [242]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.gene_ids.isin(query_data4.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.93


In [197]:
query_data1.obs.index = query_data1.obs.index+"_adams"
query_data2.obs.index = query_data2.obs.index+"_haberman"
query_data3.obs.index = query_data3.obs.index+"_reyfman_disease"
query_data4.obs.index = query_data4.obs.index+"_tsukui"

#### 2.1 Get clean metadata for each dataset

In [198]:
query_data1.obs = pd.concat([query_data1.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data1.obs.columns):]
query_data2.obs = pd.concat([query_data2.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data2.obs.columns):]
query_data3.obs = pd.concat([query_data3.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data3.obs.columns):]
query_data4.obs = pd.concat([query_data4.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data4.obs.columns):]

Rename .obs.dataset for Banovich_Kropski_2020 (haberman) and Misharin_Budinger_2018 (reyfman_disease) so that they get treated as different batch

In [199]:
query_data2.obs.dataset = query_data2.obs.dataset.cat.rename_categories({"Banovich_Kropski_2020":"Banovich_Kropski_2020_disease"})
query_data3.obs.dataset = query_data3.obs.dataset.cat.rename_categories({"Misharin_Budinger_2018":"Misharin_Budinger_2018_disease"})

In [200]:
query_data_tmp1 = query_data1.concatenate(query_data2, join="outer")
query_data_tmp2 = query_data_tmp1.concatenate(query_data3, join="outer")
ipf_data = query_data_tmp2.concatenate(query_data4, join="outer")

In [201]:
#Remove datapoints with different condition than IPF
ipf_data = ipf_data[((ipf_data.obs.condition == "IPF") | 
                    (ipf_data.obs.condition == "Healthy") | 
                    (ipf_data.obs.condition == "Control"))].copy()

In [202]:
#Set the condition "Control" to "Healthy" for consistency
ipf_data.obs.loc[ipf_data.obs.condition=="Control","condition"] = "Healthy"

In [203]:
ipf_data.obs.groupby(["condition","dataset"]).study.count()

condition  dataset                       
Control    Banovich_Kropski_2020_disease          0
           Kaminski_2020                          0
           Misharin_Budinger_2018_disease         0
           Sheppard_2020                          0
Healthy    Banovich_Kropski_2020_disease          0
           Kaminski_2020                      95303
           Misharin_Budinger_2018_disease         0
           Sheppard_2020                      27786
IPF        Banovich_Kropski_2020_disease      57668
           Kaminski_2020                     144404
           Misharin_Budinger_2018_disease     15517
           Sheppard_2020                      33817
Name: study, dtype: int64

#### 1.2 Get cell types for annotation harmonization

Sent to Lisa for cell type label harmonisation.

## 3. Nucleus data
Import and concat all raw nucleus datasets, check overlap with CellRanger Gene Filtering and Reference data (source_adata) and get clean metadata. Overlap with CellRange Genes is not too good, but was investigated - probably its a underlying difference between nucleus vs cell. Other team-members also saw overlap of 65% percent between sc data and nucleus data.

In [3]:
query_data1 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/lukassen.h5ad")
query_data2 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/duong.h5ad")
query_data3 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch1.h5ad")
query_data4 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch2.h5ad")
query_data5 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch3.h5ad")
query_data6 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch4.h5ad")

In [ ]:
query_data1.var["gene_name"] = query_data1.var.index
query_data1.var = query_data1.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data1.var.index = query_data1.var.gene_name

In [12]:
print("# genes to keep: ", np.sum(query_data1.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data1.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data1.var.index.isin(gene_names_to_keep))/len(query_data1.var.index),2))

# genes to keep:  30299
# genes total:  32738
ratio of genes to keep:  0.93


In [13]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data1.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.91


In [ ]:
query_data2.var["gene_name"] = query_data2.var.index
query_data2.var = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data2.var.index = query_data2.var.gene_name

In [15]:
print("# genes to keep: ", np.sum(query_data2.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data2.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data2.var.index.isin(gene_names_to_keep))/len(query_data2.var.index),2))

# genes to keep:  19192
# genes total:  27678
ratio of genes to keep:  0.69


In [16]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data2.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.65


In [ ]:
query_data3.var["gene_name"] = query_data3.var.index
query_data3.var = query_data3.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data3.var.index = query_data3.var.gene_name

In [18]:
print("# genes to keep: ", np.sum(query_data3.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data3.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data3.var.index.isin(gene_names_to_keep))/len(query_data3.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [19]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data3.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.64


In [ ]:
query_data4.var["gene_name"] = query_data4.var.index
query_data4.var = query_data4.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data4.var.index = query_data4.var.gene_name

In [21]:
print("# genes to keep: ", np.sum(query_data4.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data4.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data4.var.index.isin(gene_names_to_keep))/len(query_data4.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [22]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data4.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.64


In [ ]:
query_data5.var["gene_name"] = query_data5.var.index
query_data5.var = query_data5.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data5.var.index = query_data5.var.gene_name

In [24]:
print("# genes to keep: ", np.sum(query_data5.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data5.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data5.var.index.isin(gene_names_to_keep))/len(query_data5.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [25]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data4.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.64


In [ ]:
query_data6.var["gene_name"] = query_data6.var.index
query_data6.var = query_data6.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data6.var.index = query_data6.var.gene_name

In [27]:
print("# genes to keep: ", np.sum(query_data6.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data6.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data6.var.index.isin(gene_names_to_keep))/len(query_data6.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [28]:
print("Ratio of genes from reference in query: ", np.round(np.sum(source_adata.var.index.isin(query_data6.var.index))/len(source_adata.var.index),2))

Ratio of genes from reference in query:  0.64


In [29]:
query_data1.obs.index = query_data1.obs.index+"_lukassen"
query_data2.obs.index = query_data2.obs.index+"_duong"
query_data3.obs.index = query_data3.obs.index+"-sub_wang_sub_batch1"
query_data4.obs.index = query_data4.obs.index+"-sub_wang_sub_batch2"
query_data5.obs.index = query_data5.obs.index+"-sub_wang_sub_batch3"
query_data6.obs.index = query_data6.obs.index+"-sub_wang_sub_batch4"

#### 1.1 Get clean metadata for each dataset

In [224]:
query_data1.obs = pd.concat([query_data1.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data1.obs.columns):]
query_data2.obs = pd.concat([query_data2.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data2.obs.columns):]
query_data3.obs = pd.concat([query_data3.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data3.obs.columns):]
query_data4.obs = pd.concat([query_data4.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data4.obs.columns):]
query_data5.obs = pd.concat([query_data5.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data5.obs.columns):]
query_data6.obs = pd.concat([query_data6.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data6.obs.columns):]

In [225]:
query_data_tmp1 = query_data1.concatenate(query_data2, join="outer")
query_data_tmp2 = query_data_tmp1.concatenate(query_data3, join="outer")
query_data_tmp3 = query_data_tmp2.concatenate(query_data4, join="outer")
query_data_tmp4 = query_data_tmp3.concatenate(query_data5, join="outer")
nucleus_data = query_data_tmp4.concatenate(query_data6, join="outer")

#### 1.2 Get cell types for annotation harmonization

Sent to Lisa for cell type label harmonisation.

## 3. Save them

In [31]:
ipf_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Data_to_harmonize/ipf_data_to_harmonize.h5ad")

/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'dataset' as categorical
/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'ensembl-0-0-0' as categorical
/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_c

In [ ]:
nucleus_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Data_to_harmonize/nucleus_data_to_harmonize.h5ad")